# 01 — Khám phá dữ liệu CSE-CIC-IDS2018

**Notebook gốc:** Bước 0, 1, EDA ở bước 2 và phần phân bố nhãn. Đọc/ghi và chuẩn hóa schema thực hiện qua `src/data_io.py`. **Notebook có thể chạy độc lập** sau khi chép Parquet vào `data/raw/`. Không huấn luyện mô hình ở đây.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Mở thư mục project trên VS Code, hoặc chạy notebook trong notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src import config as cfg
print("Project:", PROJECT_ROOT)
print("Dataset:", cfg.DATA_DIR)


In [ ]:
import pandas as pd
from src.spark_session import create_spark
from src.utils import day_of
from src.data_io import prepare_curated
from pyspark.sql import functions as F

files = cfg.TRAIN_FILES + [cfg.VAL_FILE] + cfg.TEST_FILES
missing = [str(p) for p in files if not p.exists()]
if missing:
    raise FileNotFoundError("Chưa có file dataset trong data/raw/:\n" + "\n".join(missing))
print("Train:", [day_of(p) for p in cfg.TRAIN_FILES]); print("Val:", day_of(cfg.VAL_FILE))
print("Test:", [day_of(p) for p in cfg.TEST_FILES])
spark = create_spark()
all_df = prepare_curated(spark, files)
print("Các cột:", all_df.columns)
print("Tổng bản ghi:", all_df.count())
(cfg.OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)


In [ ]:
day_counts = all_df.groupBy("day").count().orderBy("day").toPandas()
labels = all_df.groupBy("day", "label_raw").count().orderBy("day", "label_raw").toPandas()
display(day_counts)
display(labels)
(cfg.OUT_DIR / "metrics").mkdir(parents=True, exist_ok=True)
labels.to_csv(cfg.OUT_DIR / "metrics" / "label_distribution.csv", index=False)


**Lưu ý:** Kết quả là thống kê dataset, không phải accuracy của mô hình. `prepare_curated()` có thể ghi dữ liệu lần đầu; các lần sau đặt `SKIP_INGEST=1` trước khi mở kernel để dùng lại.

In [ ]:
spark.stop()